<a href="https://colab.research.google.com/github/QuratulAin20/Mental_Health_chatbot/blob/main/db_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python -m pip install "pymongo[srv]==3.11"

In [ ]:

from pymongo.mongo_client import MongoClient
import os

uri = mongodb_url

# Create a new client and connect to the server
client = MongoClient(uri)

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [ ]:
import pandas as pd
import ast
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
import os
import json

In [ ]:
import certifi
ca = certifi.where()

In [ ]:
df = pd.read_csv("/content/dev_data.tsv", sep='\t', on_bad_lines='skip')

In [ ]:
df.head()

,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عادية خاصةً لما أكون متعلقة بأطفال كثيراً وأنا على وجه جواز أنا خايفة جداً
0,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...
1,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...
2,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...
3,هل الإحساس بقرب الاجل و الخوف من الموت و الاحل...
4,اهلا يا طبيب ، انا اعاني من عدة أعراض لمدة اسب...


In [ ]:
df.columns = ['Data']


In [ ]:
df.head()

,Data
0,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...
1,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...
2,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...
3,هل الإحساس بقرب الاجل و الخوف من الموت و الاحل...
4,اهلا يا طبيب ، انا اعاني من عدة أعراض لمدة اسب...


In [ ]:
# Basic text cleaning function
def clean_text(text):
    text = str(text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.strip().lower()
    return text


In [ ]:
# Apply cleaning
df['text'] = df['Data'].apply(clean_text)

In [ ]:
df['text']

,text
0,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...
1,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...
2,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...
3,هل الإحساس بقرب الاجل و الخوف من الموت و الاحل...
4,اهلا يا طبيب انا اعاني من عدة أعراض لمدة اسبو...
...,...
344,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...
345,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...
346,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...
347,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...


In [ ]:
df['text'] = df['text'].astype(str)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="alpcansoydas/bert-base-arabic-emotion-analysis-v2")

Device set to use cuda:0


In [ ]:
result = pipe("أنا متضايق اليوم")

In [ ]:
result

[{'label': 'fear', 'score': 0.9503520131111145}]

In [ ]:
# Predict emotion for each text entry
predicted_emotions = []
for text in df['text']:
    try:
        result = pipe(text)[0]
        predicted_emotions.append(result['label'])
    except Exception as e:
        predicted_emotions.append("unknown")


df['predicted_emotion'] = predicted_emotions


In [ ]:
df['predicted_emotion']

,predicted_emotion
0,fear
1,fear
2,sadness
3,fear
4,disgust
...,...
344,fear
345,fear
346,disgust
347,fear


In [ ]:
df.shape

(349, 3)

In [ ]:
df.head()

,Data,text,predicted_emotion
0,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,fear
1,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,fear
2,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,sadness
3,هل الإحساس بقرب الاجل و الخوف من الموت و الاحل...,هل الإحساس بقرب الاجل و الخوف من الموت و الاحل...,fear
4,اهلا يا طبيب ، انا اعاني من عدة أعراض لمدة اسب...,اهلا يا طبيب انا اعاني من عدة أعراض لمدة اسبو...,disgust


In [ ]:
df = df.drop(columns=['Data'])

In [ ]:
df.shape

(349, 2)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 349 entries, 0 to 348
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   text               349 non-null    object
 1   predicted_emotion  349 non-null    object
dtypes: object(2)
memory usage: 5.6+ KB


In [ ]:
df['predicted_emotion'].unique()

array(['fear', 'sadness', 'disgust', 'surprise', 'anger', 'joy'],
      dtype=object)

In [ ]:
df.to_csv('data.csv', index=False)

In [ ]:
import pandas as pd

# Load the labeled data (if not already in memory)
#df = pd.read_excel("labeled_dialect_data.xlsx")

# Optional: Rename columns for clarity
df = df.rename(columns={'text': 'message', 'predicted_emotion': 'emotion'})

# Convert to list of dictionaries (MongoDB expects this structure)
records = df.to_dict(orient='records')

# Save as JSON file
import json
with open("labeled_dialect_data.json", "w", encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)


In [ ]:
file_path = "/content/data.csv"

>The conversion process transforms each row of the DataFrame into a JSON object with key-value pairs, which MongoDB supports natively. This results in a list of JSON objects representing the entire dataset

>We then define the insert_data_mongodb function to insert the JSON records into a specified MongoDB database and collection.

In [ ]:
import pymongo

In [ ]:
class MentalHealthDataExtract:
    def __init__(self):
      pass
    def csv_to_json_converter(self, file_path):
       data = pd.read_csv(file_path)
       data.reset_index(drop=True, inplace=True)
       records = json.loads(data.T.to_json()).values()
       records = list(records)
       return records
    def insert_data_mongodb(self, records, database, collection):
      self.database = database
      self.collection = collection
      self.records = records
      self.mongo_client = pymongo.MongoClient(mongodb_url)
      self.database = self.mongo_client[self.database]
      self.collection = self.database[self.collection]
      self.collection.insert_many(self.records)
      return len(self.records)

In [ ]:
file_path = "/content/data.csv"
database = 'mental_health_database'
collection = 'text_emotion_data'
extractor = MentalHealthDataExtract()
records = extractor.csv_to_json_converter(file_path)
number_of_records = extractor.insert_data_mongodb(records, database, collection)
print(f'Number of records inserted: {number_of_records}')
print(records)

Number of records inserted: 349
[{'message': 'من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و ما اعرف كيف اتخلص من ذي العادة و بدت تجيني افكار بإنهاء حياتي و حاولت انتحر باكثر من مرة و أكثر من طريقة', 'emotion': 'fear', '_id': ObjectId('6880fe63535a123181f3213d')}, {'message': 'السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة لمدة طويلة من الزمن مثلا اذا سمعت شخص قام بتكرار كلمة معينة ابقى اكررها مع نفسي احاول توقيف نفسي ولكن دون جدوى حتى اصاب بلاحباط اصبت بهذا الشي من قبل والحمد لله تخلصت منه ولكنه رجع مع العلم ان الوسواس شائع في عائلتنا قالت امي استغفري بدل تكرار وشكرا', 'emotion': 'fear', '_id': ObjectId('6880fe63535a123181f3213e')}, {'message': 'اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتعامل بسلبية', 'emotion': 'sadness', '_id': ObjectId('6880fe63535a123181f3213f')}, {'message': 'هل الإحساس بقرب الاجل و الخوف من الموت و الاحلام من اعراض الاكتئاب و القلق و كيف يمكنني تخطي هذه المرحلة لان حياتي اصبحت جحيم', 'emotion': 'fear', '_id': ObjectId('6880fe63535a123181f32140')}, {'message': 